### **✏️ Exercises**
- Use LangChain's docling integration for document ingestion
- Rewrite the whole pipeline using LangChain

In [14]:
# Install Dependencies
!pip install -q --upgrade langchain-docling langchain-community langchain-text-splitters langchain-qdrant langchain-groq langchain-huggingface langchain


 ## Document loading and chunking

In [15]:
from langchain_docling import DoclingLoader

FILE_PATH = "https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/AtliqAI_HR_Policies.pdf"

# Initialize the DoclingLoader
loader = DoclingLoader(file_path=FILE_PATH)

# Load the documents into LangChain format. The DoclingLoader directly returns
# a list of LangChain Document objects, which can be thought of as initial 'chunks'.
langchain_docs = loader.load()

print(f"Loaded {len(langchain_docs)} documents/initial chunks using LangChain's DoclingLoader.")
print("First document/chunk's page content sample:")
print(langchain_docs[0].page_content[:500])
print("\nFirst document/chunk's metadata:")
print(langchain_docs[0].metadata)

[INFO] 2026-06-10 03:30:59,819 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-10 03:30:59,823 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-10 03:31:00,040 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-10 03:31:00,044 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-10 03:31:00,348 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-10 03:31:00,350 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-10 03:31:00,370 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-06-10 03:31:00,371 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-06-10 03:31:00,469 [Ra

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Loaded 44 documents/initial chunks using LangChain's DoclingLoader.
First document/chunk's page content sample:
AtliqAI HR Policies
AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.

First document/chunk's metadata:
{'source': 'https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/AtliqAI_HR_Policies.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/1', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 33.301339386, 't': 776.9694969176504, 'r': 548.6781593722308, 'b': 750.5246198306972, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 325]}]}

Now that the document is loaded as a LangChain `Document` object, you can further process it, for example, by splitting it into smaller chunks using a text splitter.

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
langchain_chunks = text_splitter.split_documents(langchain_docs)

print(f"Split into {len(langchain_chunks)} chunks.")
print("First chunk's content sample:")
print(langchain_chunks[0].page_content[:500])

Split into 44 chunks.
First chunk's content sample:
AtliqAI HR Policies
AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.


## Embedding and Indexing


In [17]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

# Select your embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Initialize Qdrant Client
# Use location=":memory:" for instant testing
vectorstore = QdrantVectorStore.from_documents(
    documents=langchain_chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="docling_rag"
)

print("LangChain Qdrant vector store initialized.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LangChain Qdrant vector store initialized.


Now that the vector store is set up, we can create a LangChain retriever and an LLM to build a RetrievalQA chain.

## Retrieval and Generation

In [18]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

import os
from google.colab import userdata

# Safely extract your key from the Colab secrets manager
# Note: Ensure the secret name matches exactly what you typed on the left panel (e.g., "GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Define the GROQ model
GROQ_MODEL  = "openai/gpt-oss-safeguard-20b"

# Create a LangChain retriever
langchain_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Initialize the LangChain LLM (ChatGroq)
# Ensure GROQ_API_KEY is set in environment or Colab secrets
langchain_llm = ChatGroq(temperature=0.2, model_name=GROQ_MODEL)


# Function to print context and question
def print_context_and_question(input_dict):
    print("\n--- Context ---")
    for doc in input_dict["context"]:
        print(doc.page_content)
    print("\n--- Question ---")
    print(input_dict["question"])
    return input_dict

# Design your AI prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context.
Context: {context}
Question: {question}
""")

# Construct LCEL pipeline
rag_chain = (
    {"context": langchain_retriever, "question": RunnablePassthrough()}
    | RunnableLambda(print_context_and_question) # Added to print context and question
    | prompt
    | langchain_llm
    | StrOutputParser()
)

# Run a live query
response = rag_chain.invoke("What are the key points?")
print(response)



--- Context ---
AtliqAI HR Policies
AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.
Rating Scale
Performance is rated on a 5-point scale: 1 - Needs Improvement, 2 - Partially Meets Expectations, 3 - Meets Expectations, 4 - Exceeds Expectations, 5 - Outstanding. Employees rated 1 for two consecutive review cycles will be placed on a Performance Improvement Plan (PIP).
Exit Interview
All exiting employees are encouraged to participate in a confidential exit interview with the HR team. The purpose of the exit interview is to gather feedback on the employee's experience at AtliqAI, identify areas of improvement, and ensure a smooth transition. Participation is voluntary but strongly recommended.
Performance Improvem

In [19]:
response = rag_chain.invoke("Tell me the leave policy.")
print(response)


--- Context ---
Leave Without Pay
Employees who exhaust all available leave balances may apply for leave without pay (LWP). LWP must be approved by the reporting manager and HR. More than 10 days of LWP in a financial year may impact annual performance ratings and bonus eligibility.
Casual Leave
Every confirmed employee is entitled to 12 casual leaves per calendar year, credited at 1 leave per month. Casual leave can be availed for personal errands, minor illness, or unplanned absences. A maximum of 3 consecutive casual leaves can be taken at a time. Casual leaves cannot be carried forward to the next calendar year and lapse on December 31st.
Sick Leave
Employees are entitled to 10 sick leaves per calendar year. Sick leave can be availed in case of illness, hospitalisation, or medical procedures. A medical certificate from a registered practitioner is mandatory for sick leave of more than 2 consecutive days. Unused sick leaves up to a maximum of 10 can be carried forward to the follow